# The Complete Plotting Tutorial: All the plots you need for modern Machine Learning

Welcome to the analytical core of Data Science. In this tutorial, we will cover the five essential plots every Machine Learning engineer must master. 

### The Analytical Goals of ML Plotting:
1. **Distributions:** Are my features normally distributed, skewed, or bimodal?
2. **Relationships:** Do these two continuous variables have a linear correlation?
3. **Categorical Separation:** Does this categorical feature effectively split my target variable?
4. **Multicollinearity:** Are my features highly correlated with each other?
5. **Model Evaluation:** Is my model learning or overfitting over time?

We will use **Seaborn**, which is built on top of Matplotlib but provides high-level mathematical aggregations and stunning aesthetics right out of the box.

In [ ]:
# Cell 2: Environment Setup and Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set the analytical aesthetic theme for all plots
sns.set_theme(style="whitegrid", palette="muted")

print("Plotting libraries successfully loaded!")

## Step 1: Generating Analytical ML Data

To demonstrate these plots, we need realistic data. We will synthesize a dataset representing a classic ML problem: Predicting a person's **Income** (Continuous Target) and whether they will **Default on a Loan** (Categorical Target) based on their Age, Years of Experience, and Credit Score.

In [ ]:
# Cell 4: Data Generation

np.random.seed(42)
n_samples = 500

# 1. Independent Variables
age = np.random.normal(loc=35, scale=10, size=n_samples).astype(int)
experience = np.clip(age - 22 + np.random.normal(0, 2, n_samples), 0, None)
credit_score = np.random.normal(loc=650, scale=50, size=n_samples).astype(int)

# 2. Continuous Target: Income (Highly correlated with experience)
income = 40000 + (experience * 5000) + np.random.normal(0, 15000, n_samples)

# 3. Categorical Target: Default (Binary)
# People with lower credit scores and lower income are mathematically more likely to default
logit = -5.0 + (0.01 * (800 - credit_score)) + (0.00005 * (100000 - income))
probability = 1 / (1 + np.exp(-logit))
default = np.random.binomial(1, probability)

# Assemble DataFrame
df = pd.DataFrame({
    'Age': age,
    'Experience': experience,
    'Credit_Score': credit_score,
    'Income': income,
    'Default': default
})

# Map binary 0/1 to human-readable categories for plotting
df['Default_Label'] = df['Default'].map({0: 'No', 1: 'Yes'})

print("Synthetic ML Dataset Generated.")
display(df.head())

## Plot 1: Scatter Plots & Regression Lines (Relationships)

When analyzing two continuous variables, the **Scatter Plot** is king. It allows us to visually detect linear, exponential, or polynomial relationships.

In ML, we often overlay an Ordinary Least Squares (OLS) regression line to immediately see the mathematical trend and the confidence interval (the shaded region around the line).

In [ ]:
# Cell 6: Scatter Plot with Regression Line

plt.figure(figsize=(10, 6))

# sns.regplot automatically calculates and plots the linear regression line of best fit
sns.regplot(data=df, x='Experience', y='Income', 
            scatter_kws={'alpha':0.5, 'edgecolor':'k'}, 
            line_kws={'linewidth':3})

plt.title("Income vs. Years of Experience (Linear Relationship)", fontsize=16)
plt.xlabel("Years of Experience")
plt.ylabel("Income ($)")

# Format y-axis with dollar signs (optional but highly recommended for analysis)
plt.gca().yaxis.set_major_formatter(plt.matplotlib.ticker.StrMethodFormatter('${x:,.0f}'))

plt.show()
print("Analytical Note: We observe a strong positive linear correlation. Experience will be an excellent predictor feature for Income.")

## Plot 2: Histograms & KDE (Distributions)

Before feeding data into algorithms like Linear Regression or Gaussian Naive Bayes, you must know if your features are normally distributed (bell-shaped) or skewed.

We use a **Histogram** overlaid with a **Kernel Density Estimate (KDE)**. The KDE mathematically smooths the discrete histogram bins into a continuous probability density curve.

In [ ]:
# Cell 8: Distribution Plot

plt.figure(figsize=(10, 6))

# sns.histplot with kde=True provides both the discrete counts and continuous probability curve
sns.histplot(data=df, x='Credit_Score', hue='Default_Label', 
             kde=True, bins=30, alpha=0.6, element="step")

plt.title("Distribution of Credit Scores grouped by Loan Default", fontsize=16)
plt.xlabel("Credit Score")
plt.ylabel("Frequency")

plt.show()
print("Analytical Note: We see two distinct overlapping Gaussian distributions. Those who Default (Yes) clearly cluster around a lower mean credit score.")

## Plot 3: Box Plots (Categorical Comparison & Outlier Detection)

The **Box Plot** is the most mathematically dense plot in data science. It visualizes the Interquartile Range (IQR). 
* The box represents the middle 50% of the data ($Q_1$ to $Q_3$).
* The line in the middle is the Median.
* The "whiskers" extend to $1.5 \times IQR$.
* Any dots outside the whiskers are mathematical **Outliers**.

We use this to see if a categorical feature cleanly separates a continuous feature.

In [ ]:
# Cell 10: Box Plot for Outliers and Separation

plt.figure(figsize=(8, 6))

sns.boxplot(data=df, x='Default_Label', y='Income')

plt.title("Income Distribution by Default Status (Box Plot)", fontsize=16)
plt.xlabel("Did they Default?")
plt.ylabel("Income ($)")
plt.gca().yaxis.set_major_formatter(plt.matplotlib.ticker.StrMethodFormatter('${x:,.0f}'))

plt.show()
print("Analytical Note: The medians are significantly different, but there are numerous high-income outliers in the 'Yes' category.")

## Plot 4: The Correlation Heatmap (Feature Selection)

Feeding highly correlated features (e.g., Age and Experience) into certain ML models (like Linear Regression) causes **Multicollinearity**, mathematically destabilizing the model's coefficients.

We calculate the Pearson Correlation Coefficient matrix (ranging from -1 to 1) and plot it as a color-coded **Heatmap**.

In [ ]:
# Cell 12: Correlation Matrix Heatmap

plt.figure(figsize=(8, 6))

# Calculate the correlation matrix (dropping the string label column)
corr_matrix = df.drop(columns=['Default_Label']).corr()

# Plot the heatmap
# annot=True prints the mathematical values inside the squares
# cmap='coolwarm' uses blue for negative correlation and red for positive correlation
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)

plt.title("Feature Correlation Heatmap", fontsize=16)
plt.show()
print("Analytical Note: Age and Experience have a correlation of ~0.97. For linear models, we should likely drop one of these features to prevent multicollinearity.")

## Plot 5: Line Plots (Learning Curves & Time Series)

The **Line Plot** is used when there is a sequential relationship on the X-axis (like Time, or Epochs during neural network training). 

Let's simulate tracking a Deep Learning model's Loss Function over 100 epochs to ensure it is actually learning and not overfitting.

In [ ]:
# Cell 14: Learning Curve (Line Plot)

# Simulate Neural Network training data
epochs = np.arange(1, 101)
# Training loss decreases logarithmically
train_loss = 2.0 * np.exp(-epochs / 20) + 0.1 + np.random.normal(0, 0.02, 100)
# Validation loss decreases, then starts increasing (simulating overfitting)
val_loss = 2.0 * np.exp(-epochs / 20) + 0.15 + (epochs / 150)**2 + np.random.normal(0, 0.02, 100)

plt.figure(figsize=(10, 6))

# Plot both lines on the same axes
sns.lineplot(x=epochs, y=train_loss, label='Training Loss', linewidth=2)
sns.lineplot(x=epochs, y=val_loss, label='Validation Loss', linewidth=2, linestyle='--')

# Draw a vertical line where the model starts to overfit
optimal_epoch = np.argmin(val_loss)
plt.axvline(x=optimal_epoch, color='red', alpha=0.5, label=f'Optimal Stop (Epoch {optimal_epoch})')

plt.title("Model Evaluation: Learning Curves", fontsize=16)
plt.xlabel("Training Epoch")
plt.ylabel("Categorical Crossentropy Loss")
plt.legend()

plt.show()

## Conclusion

You now possess the foundational visual toolkit of a Data Scientist.

1.  **Scatter Plots** reveal raw relationships.
2.  **Histograms** expose the shape and mathematical distribution of your data.
3.  **Box Plots** relentlessly identify outliers and class separation.
4.  **Heatmaps** prevent you from feeding redundant data to your models.
5.  **Line Plots** prove whether your algorithms are actually learning over time.

By utilizing `seaborn` and `matplotlib`, you can transform raw numbers into actionable, analytical business intelligence!